In [1]:
!cd ../.. && pip install -v . --group=dev --group=observability --group=analysis

Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
Processing /home/jovyan/work
  Running command installing build dependencies
  Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
    Obtaining dependency information for setuptools>=40.8.0 from https://files.pythonhosted.org/packages/9d/76/f789f7a86709c6b087c5a2f52f911838cad707cc613162401badc665acfe/setuptools-82.0.1-py3-none-any.whl.metadata
    Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Installing build dependencies ... done
  Running command Getting requirements to build wheel
  running egg_info
  writing massi.egg-info/PKG-INFO
  writing dependency_links to massi.egg-info/dependency_links.txt
  writing requirements to massi.egg-info/requires.txt
  writing top-level names to massi.egg-info/top_level.txt
  reading manifest file 'massi.egg-info/SOURCES.txt'
  writing manifest file 'massi.egg-info/S

# Overview of Experiment results

In [27]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from lxml import etree

import itertools as ittls
import ast 
import pprint
from pathlib import Path
import json

In [3]:
experiment_results = pd.read_csv("../runtime/experiment_cross_scenarios_cases.log.csv")

In [22]:
gbif_cases = pd.read_csv("../runtime/full_gbif_cases.csv")

In [4]:
from typing import TypedDict

class MetricSummary(TypedDict):
    mean: float
    median: float
    min: float
    max: float
    std: float

def get_summary(series: pd.Series) -> MetricSummary:
    """Computes standard descriptive statistics for a numeric series."""
    return {
        "mean": float(series.mean()),
        "median": float(series.median()),
        "min": float(series.min()),
        "max": float(series.max()),
        "std": float(series.std())
    }

In [10]:
class MetadataFileStats(TypedDict):
    schema: str
    
    grand_nr_elements: MetricSummary
    grand_nr_unique_elements: MetricSummary
    group_nr_elements: MetricSummary
    group_nr_unique_elements: MetricSummary
    total_nr_unique_elements: int
    
    grand_nr_attributes: MetricSummary
    grand_nr_unique_attributes: MetricSummary
    group_nr_attributes: MetricSummary
    group_nr_unique_attributes: MetricSummary
    total_nr_unique_attributes: int

    grand_nr_text_nodes: MetricSummary
    group_nr_text_nodes: MetricSummary

    grand_element_depth: MetricSummary
    group_element_depth: MetricSummary

    
    grand_element_fanout_degree: MetricSummary
    group_element_fanout_degree: MetricSummary
    
    grand_element_to_attribute_degree: MetricSummary
    group_element_to_attribute_degree: MetricSummary

    grand_text_content_length: MetricSummary
    group_text_content_length: MetricSummary

In [55]:
def is_hidden_file(fn: str) -> bool:
    return any((
        fn.startswith("__"),  # reserved/hidden files, e.g. __MACOSX
        fn.startswith("."),   # hidden files UNIX
    ))

def is_metadata_xml(fn: str) -> bool:
    """Check for xml but be aware of zip artifacts"""
    return fn.endswith(".xml") and not is_hidden_file(fn)

In [14]:
EXPERIMENT_BASE_DIR = "../../_data/gbif_data_archive/"

def grounded_path(suffix: str | Path) -> Path:
    return Path(EXPERIMENT_BASE_DIR) / suffix

In [59]:
def parse_file_to_overview(path: str | Path) -> dict:
    if not path: return 
    try:
        tree = etree.parse(path)
        for node in tree.iter():
            attrs = node.attrib
            text = node.text.strip() if node.text else ""

            return {
                "file_path": path,
                "tag": node.tag,
                "depth": sum(1 for _ in node.iterancestors()),
                "nr_attributes": len(attrs),
                "nr_unique_attributes": len(set(attrs.keys())),
                "fanout": len(node),
                "text_length": len(text),
                "attr_names": set(attrs.keys()),
                "is_text_node": bool(text) ,
                "text_content": text,
            }
    except Exception as e:
        print(f"Skipping {path} due to error: {e}")


def parse_xml_to_overview(xml_data: str) -> dict:
    if not xml_data_series: return {}
    try:
        tree = etree.fromstring(xml_data_series)
        for node in tree.iter():
            attrs = node.attrib
            text = node.text.strip() if node.text else ""

            return {
                "file_path": path,
                "tag": node.tag,
                "depth": sum(1 for _ in node.iterancestors()),
                "nr_attributes": len(attrs),
                "nr_unique_attributes": len(set(attrs.keys())),
                "fanout": len(node),
                "text_length": len(text),
                "attr_names": set(attrs.keys()),
                "is_text_node": bool(text) ,
                "text_content": text,
            }
    except Exception as e:
        print(f"Skipping {path} due to error: {e}")

In [80]:
def create_metadata_dataframes(el_df: pd.Series):    
    file_df = el_df.groupby(level=0).agg(
        nr_elements=('tag', 'count'),
        nr_unique_elements=('tag', 'nunique'),
        total_attributes=('nr_attributes', 'sum'),
        avg_depth=('depth', 'mean'),
        max_depth=('depth', 'max'),
        avg_fanout=('fanout', 'mean'),
        total_text_length=('text_length', 'sum'),
        nr_text_nodes=('text_length', lambda x: (x > 0).sum())
    ).reset_index()
    
    # Add calculated density columns for plotting
    file_df['attr_density'] = file_df['total_attributes'] / file_df['nr_elements']
    
    return el_df, file_df

In [61]:
def get_metadata_file_stats(el_df: pd.DataFrame, file_df: pd.DataFrame, schema: str) -> MetadataFileStats:
    # Grand (Pooled) - using Element DF
    total_unique_elements = el_df['tag'].nunique()
    total_unique_attrs = len(set().union(*el_df['attr_names']))
    
    # Group (Distribution of file properties) - using File DF
    return {
        "schema": schema,
        
        # Elements
        "grand_nr_elements": get_summary(file_df['nr_elements']), 
        "grand_nr_unique_elements": get_summary(file_df['nr_unique_elements']),
        "group_nr_elements": get_summary(file_df['nr_elements']), 
        "group_nr_unique_elements": get_summary(file_df['nr_unique_elements']),
        "total_nr_unique_elements": total_unique_elements,
        
        # Attributes
        "grand_nr_attributes": get_summary(file_df['total_attributes']),
        "grand_nr_unique_attributes": get_summary(el_df.groupby("file_path")['nr_unique_attributes'].mean()), 
        "group_nr_attributes": get_summary(el_df.groupby("file_path")['nr_attributes'].mean()),
        "total_nr_unique_attributes": total_unique_attrs,

        # Structure & Content
        "grand_element_depth": get_summary(el_df['depth']),  # Pooled
        "group_element_depth": get_summary(file_df['avg_depth']), # Mean of means
        
        "grand_element_fanout_degree": get_summary(el_df['fanout']),
        "group_element_fanout_degree": get_summary(file_df['avg_fanout']),

        "grand_text_content_length": get_summary(el_df[el_df['text_length'] > 0]['text_length']),
        "group_text_content_length": get_summary(file_df['total_text_length']),
        
        "grand_nr_text_nodes": get_summary(file_df['nr_text_nodes']),
        "group_nr_text_nodes": get_summary(file_df['nr_text_nodes']), # Summarizing the counts
        
        "grand_element_to_attribute_degree": get_summary(el_df['nr_attributes']),
        "group_element_to_attribute_degree": get_summary(file_df['attr_density']),
    }

In [62]:
gbif_cases["paper_download_info_dict"] = gbif_cases["paper_download_info"].apply(ast.literal_eval)
gbif_cases["dwca_zip_info_dict"] = gbif_cases["dwca_zip_info"].apply(ast.literal_eval)

In [63]:
gt_eml_files = gbif_cases["ground_truth_metadata_eml"]
gt_dwc_files = gbif_cases["ground_truth_metadata_dwc"]

In [81]:
gt_eml_element_overviews, gt_eml_file_overviews = create_metadata_dataframes(gt_eml_files.apply(grounded_path).apply(parse_file_to_overview).apply(pd.Series))
gt_dwc_element_overviews, gt_dwc_file_overviews = create_metadata_dataframes(gt_dwc_files.apply(grounded_path).apply(parse_file_to_overview).apply(pd.Series))

In [84]:
gt_dwc_element_overviews.describe(include="all")

,file_path,tag,depth,nr_attributes,nr_unique_attributes,fanout,text_length,attr_names,is_text_node,text_content
count,179,179,179.0,179.00000,179.00000,179.000000,179.0,179,179,179
unique,179,1,NaN,NaN,NaN,NaN,NaN,2,1,1
top,../../_data/gbif_data_archive/ground_truth/dat...,{http://rs.tdwg.org/dwc/text/}archive,NaN,NaN,NaN,NaN,NaN,{metadata},False,
freq,1,179,NaN,NaN,NaN,NaN,NaN,176,179,179
mean,NaN,NaN,0.0,0.98324,0.98324,1.558659,0.0,NaN,NaN,NaN
std,NaN,NaN,0.0,0.12873,0.12873,0.695629,0.0,NaN,NaN,NaN
min,NaN,NaN,0.0,0.00000,0.00000,1.000000,0.0,NaN,NaN,NaN
25%,NaN,NaN,0.0,1.00000,1.00000,1.000000,0.0,NaN,NaN,NaN
50%,NaN,NaN,0.0,1.00000,1.00000,1.000000,0.0,NaN,NaN,NaN
75%,NaN,NaN,0.0,1.00000,1.00000,2.000000,0.0,NaN,NaN,NaN


In [85]:
gt_dwc_file_overviews.describe(include="all")

,index,nr_elements,nr_unique_elements,total_attributes,avg_depth,max_depth,avg_fanout,total_text_length,nr_text_nodes,attr_density
count,179.000000,179.0,179.0,179.00000,179.0,179.0,179.000000,179.0,179.0,179.00000
mean,89.000000,1.0,1.0,0.98324,0.0,0.0,1.558659,0.0,0.0,0.98324
std,51.816986,0.0,0.0,0.12873,0.0,0.0,0.695629,0.0,0.0,0.12873
min,0.000000,1.0,1.0,0.00000,0.0,0.0,1.000000,0.0,0.0,0.00000
25%,44.500000,1.0,1.0,1.00000,0.0,0.0,1.000000,0.0,0.0,1.00000
50%,89.000000,1.0,1.0,1.00000,0.0,0.0,1.000000,0.0,0.0,1.00000
75%,133.500000,1.0,1.0,1.00000,0.0,0.0,2.000000,0.0,0.0,1.00000
max,178.000000,1.0,1.0,1.00000,0.0,0.0,4.000000,0.0,0.0,1.00000


## Metadata Structure

In [30]:
experiment_results["outcome_dict"] = experiment_results["outcome_json"].apply(json.loads)

In [48]:
successful_experiments = experiment_results[experiment_results["success"]]

In [51]:
successful_experiments["outcome_metadata_xml"] = successful_experiments["outcome_dict"].apply(lambda d: d['resolutions'][0]['working_copy_xml_blueprint'])

In [86]:
successful_experiments_eml = successful_experiments[successful_experiments["schema"] == "eml"]

In [88]:
successful_experiments_dwc = successful_experiments[successful_experiments["schema"] == "dwc"]